In [2]:
import pandas as pd
import glob
import os

In [7]:
caiso = pd.read_csv('../caiso_dam_lmp_parallel_mit.csv')

In [5]:
missing_CAISO = "../Missing_CAISO"
missing_files = glob.glob(os.path.join(missing_CAISO, "*.csv"))

dfs_missing = []

for fp in missing_files:
    df = pd.read_csv(fp)
    dfs_missing.append(df)

missing = pd.concat(dfs_missing, ignore_index=True)

In [8]:
combined = pd.concat([caiso, missing], ignore_index=True)

In [13]:
combined.head()

,INTERVALSTARTTIME_GMT,INTERVALENDTIME_GMT,OPR_DT,OPR_HR,OPR_INTERVAL,NODE_ID_XML,NODE_ID,NODE,MARKET_RUN_ID,LMP_TYPE,XML_DATA_ITEM,PNODE_RESMRID,GRP_TYPE,POS,MW,GROUP
0,2023-12-31T09:00:00-00:00,2023-12-31T10:00:00-00:00,2023-12-31,2,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,42.43584,1
1,2024-01-01T06:00:00-00:00,2024-01-01T07:00:00-00:00,2023-12-31,23,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,46.36527,1
2,2023-12-31T11:00:00-00:00,2023-12-31T12:00:00-00:00,2023-12-31,4,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,39.70369,1
3,2023-12-31T12:00:00-00:00,2023-12-31T13:00:00-00:00,2023-12-31,5,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,40.51819,1
4,2023-12-31T10:00:00-00:00,2023-12-31T11:00:00-00:00,2023-12-31,3,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,40.23057,1


In [41]:
df_new = pd.DataFrame()

df_new['timestamp_utc'] = pd.to_datetime(combined['INTERVALENDTIME_GMT'], errors='coerce')
df_new['Location Name'] = combined['NODE']
df_new['Location Type'] = 'Node'
df_new['LMP'] = combined['MW']
df_new['MCC'] = pd.NA  
df_new['MLC'] = pd.NA  

df_new = df_new.sort_values('timestamp_utc').reset_index(drop=True)

start_utc = pd.Timestamp("2024-01-01 08:00:00", tz="UTC")
end_utc   = pd.Timestamp("2025-01-01 09:00:00", tz="UTC")

df_final = df_new[
    (df_new["timestamp_utc"] >= start_utc) &
    (df_new["timestamp_utc"] < end_utc)
].copy()

In [42]:
df_final = df_final.set_index('timestamp_utc')
df_final.index.min(), df_final.index.max()

(Timestamp('2024-01-01 08:00:00+0000', tz='UTC'),
 Timestamp('2025-01-01 08:00:00+0000', tz='UTC'))

In [43]:
expected_index = pd.date_range(
    start="2024-01-01 08:00:00",
    end="2025-01-01 08:00:00",
    freq="h",
    tz="UTC",
    inclusive="left"   # include start, exclude end
)
missing_hours = expected_index.difference(df_final.index)
missing_hours


DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='h')

In [44]:
df_sdge = (
    df_final
    .reset_index()
    .query("`Location Name` == 'DLAP_SDGE-APND'")
    .set_index("timestamp_utc")
)

In [45]:
df_sdge

,Location Name,Location Type,LMP,MCC,MLC
timestamp_utc,,,,,
2024-01-01 08:00:00+00:00,DLAP_SDGE-APND,Node,46.05730,<NA>,<NA>
2024-01-01 09:00:00+00:00,DLAP_SDGE-APND,Node,46.56375,<NA>,<NA>
2024-01-01 10:00:00+00:00,DLAP_SDGE-APND,Node,45.72695,<NA>,<NA>
2024-01-01 11:00:00+00:00,DLAP_SDGE-APND,Node,45.49123,<NA>,<NA>
2024-01-01 12:00:00+00:00,DLAP_SDGE-APND,Node,45.12528,<NA>,<NA>
...,...,...,...,...,...
2025-01-01 04:00:00+00:00,DLAP_SDGE-APND,Node,56.97198,<NA>,<NA>
2025-01-01 05:00:00+00:00,DLAP_SDGE-APND,Node,57.02665,<NA>,<NA>
2025-01-01 06:00:00+00:00,DLAP_SDGE-APND,Node,52.82632,<NA>,<NA>


In [46]:
df_sdge.index.nunique()
df_sdge = df_sdge[~df_sdge.index.duplicated(keep='first')]
df_sdge.index.nunique(), len(df_sdge)
dupes = df_sdge.index[df_sdge.index.duplicated()]
dupes


DatetimeIndex([], dtype='datetime64[ns, UTC]', name='timestamp_utc', freq=None)

In [47]:
start_utc = pd.Timestamp("2024-01-01 08:00:00", tz="UTC")
end_utc   = pd.Timestamp("2025-01-01 08:00:00", tz="UTC")

df_sdge = df_sdge[(df_sdge.index >= start_utc) & (df_sdge.index < end_utc)]



In [48]:
len(df_sdge), df_sdge.index.nunique()

(8784, 8784)

In [49]:
df_sdge_clean = df_sdge.drop(columns=["Location Name", "Location Type", "MCC", "MLC"])

In [50]:
df_sdge_clean.index.nunique(), len(df_sdge_clean)

(8784, 8784)

In [51]:
df_sdge_clean.to_csv("../data/sdge_2024_lmp.csv")